In [0]:
%pip install python-dotenv==1.0.1 -q

In [0]:
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("entity", "payments")
dbutils.widgets.text("max_pages", "0")  # 0 = read all pages
dbutils.widgets.dropdown("full_refresh", "no", ["no", "yes"])

In [0]:
import os
import sys
import time
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F

repo_root = os.path.dirname(os.getcwd())  # this notebook lives in <repo>/notebooks
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

os.environ["API_BASE_URL"] = "https://ev-project-navy-mu.vercel.app"
os.environ["API_USERNAME"] = dbutils.secrets.get("voltgrid", "api-username")
os.environ["API_PASSWORD"] = dbutils.secrets.get("voltgrid", "api-password")
os.environ["PII_SALT"] = dbutils.secrets.get("voltgrid", "pii-salt")
os.environ["DB_PASSWORD"] = "not-used-on-databricks"

from config.settings import API_BASE_URL, LOOKBACK_MIN
from ingestion.extractor import ENDPOINTS, fetch_pages
from ingestion.http_client import login, make_session
from ingestion.raw_writer import raw_page_path, write_jsonl
from ingestion.watermark import max_updated_at, since_param

RAW_ROOT = "/Volumes/voltgrid/raw/api_landing"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]

In [0]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS voltgrid.raw.pipeline_watermarks (
        entity STRING, watermark STRING, updated_at TIMESTAMP)
""")
spark.sql("""
    CREATE TABLE IF NOT EXISTS voltgrid.raw.run_audit (
        run_id STRING, entity STRING, load_type STRING, status STRING,
        page_count INT, row_count BIGINT, since STRING, seconds DOUBLE,
        error STRING, finished_at TIMESTAMP)
""")


def get_watermark(entity):
    row = spark.sql(
        "SELECT watermark FROM voltgrid.raw.pipeline_watermarks WHERE entity = :e",
        args={"e": entity},
    ).first()
    return row["watermark"] if row else None


def save_watermark(entity, value):
    spark.sql(
        """
        MERGE INTO voltgrid.raw.pipeline_watermarks t
        USING (SELECT :e AS entity, :w AS watermark, current_timestamp() AS updated_at) s
        ON t.entity = s.entity
        WHEN MATCHED THEN UPDATE SET t.watermark = s.watermark, t.updated_at = s.updated_at
        WHEN NOT MATCHED THEN INSERT *
        """,
        args={"e": entity, "w": value},
    )

In [0]:
entity = dbutils.widgets.get("entity")
max_pages = int(dbutils.widgets.get("max_pages")) or None
full_refresh = dbutils.widgets.get("full_refresh") == "yes"

watermark = None if full_refresh else get_watermark(entity)
since = since_param(watermark, LOOKBACK_MIN)
params = {"updated_after": since} if since else {}
load_type = "incremental" if since else "full"
ingest_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
folder = raw_page_path(RAW_ROOT, "voltgrid", entity, ingest_date)
print(f"{entity}: {load_type} load, since={since}, max_pages={max_pages}")

session = make_session(login())
t0 = time.time()
pages = rows = 0
new_watermark = watermark
status, error = "SUCCESS", None
try:
    for page, records in fetch_pages(
        session, API_BASE_URL, ENDPOINTS[entity],
        page_size=1000, params=params, max_pages=max_pages,
    ):
        write_jsonl(folder, f"{run_id}_p{page:05d}.jsonl", records)
        pages += 1
        rows += len(records)
        new_watermark = max_updated_at(records, new_watermark)
    # Save the watermark only after a COMPLETE, successful load
    if max_pages is None and new_watermark:
        save_watermark(entity, new_watermark)
except Exception as exc:
    status, error = "FAILED", str(exc)[:500]
    raise
finally:
    audit = spark.createDataFrame(
        [(run_id, entity, load_type, status, pages, rows, since, round(time.time() - t0, 1), error)],
        "run_id string, entity string, load_type string, status string, "
        "page_count int, row_count long, since string, seconds double, error string",
    ).withColumn("finished_at", F.current_timestamp())
    audit.write.mode("append").saveAsTable("voltgrid.raw.run_audit")

print(f"{status}: {pages} pages, {rows} rows, watermark now {new_watermark}")

In [0]:
display(spark.sql("SELECT * FROM voltgrid.raw.run_audit ORDER BY finished_at DESC"))
display(spark.sql("SELECT * FROM voltgrid.raw.pipeline_watermarks"))
display(dbutils.fs.ls(str(folder)))
print("rows in today's raw folder:", spark.read.json(str(folder)).count())

In [0]:
dbutils.fs.rm(str(folder), True)
print("cleared:", folder)